# Chess Learning — Stage 02: Self-Play & TD Learning (Neural Net)

**Condition:** NEURAL_NET — same 19-feature input as the learnable material condition; evaluator is a single-hidden-layer MLP (19 → 32 → 1, tanh), trained with TD(0) + Adam.

| Component | Description |
|---|---|
| **Engine** | `engine_nn.py` — ChessNet MLP; shares feature extraction with `engine_learnable_material` |
| **Self-play** | Epsilon-greedy; both sides played by the same model |
| **Learning** | TD(0) bootstrapping with Adam optimizer; per-move online updates |
| **Logging** | Attribution weights (dV/dx_i at starting position) checkpointed to DB |


## Setup

In [12]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "chess", "-q"])


0

In [13]:
try:
    import torch
    print(f"PyTorch {torch.__version__} already available")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "-q"])
    import torch
    print(f"PyTorch {torch.__version__} installed")


PyTorch 2.14.1+cpu already available


In [14]:
import chess
import chess.pgn
import random
import sqlite3
import time
import os
import torch
import torch.nn as nn
from datetime import datetime
from collections import defaultdict


### Import engine

This notebook uses `engine_nn.py`. Feature extraction is shared with
`engine_learnable_material.py`; only the evaluator changes.


In [15]:
sys.path.insert(0, os.path.dirname(os.path.abspath("selfplay.ipynb")))
from engine_nn import (
    FEATURE_NAMES, N_FEATURES,
    new_model, save_model, load_model,
    attribution_weights,
    extract_features,
    get_best_move,
)
print(f"Engine loaded. Features ({N_FEATURES}):", FEATURE_NAMES)
print("PyTorch version:", torch.__version__)


Engine loaded. Features (19): ['material_queen', 'material_rook', 'material_bishop', 'material_knight', 'material_pawn', 'passed_pawn', 'doubled_pawn', 'isolated_pawn', 'backward_pawn', 'king_safety', 'center_control', 'rook_open_file', 'connected_rooks', 'knight_outpost', 'bishop_pair', 'rook_seventh', 'piece_development', 'mobility', 'pawn_advancement']
PyTorch version: 2.14.1+cpu


## Database Setup

Identical schema to the learnable material condition. The `weights` table stores
attribution weights (dV/dx_i at starting position) instead of raw parameters,
enabling the same R analysis scripts to work across conditions.


In [16]:
def setup_database(db_path):
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS runs (
            run_id        INTEGER PRIMARY KEY,
            started_at    TEXT,
            depth         INTEGER,
            learning_rate REAL,
            move_limit    INTEGER,
            notes         TEXT
        );
        CREATE TABLE IF NOT EXISTS games (
            game_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            n_halfmoves   INTEGER,
            duration_s    REAL,
            terminated_by TEXT
        );
        CREATE TABLE IF NOT EXISTS weights (
            weight_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            weight_value  REAL
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            stat_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            feature_name  TEXT,
            mean_value    REAL
        );
        CREATE TABLE IF NOT EXISTS game_records (
            record_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            pgn           TEXT,
            td_error      REAL
        );
        CREATE TABLE IF NOT EXISTS weight_deltas (
            delta_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            delta         REAL
        );
        CREATE TABLE IF NOT EXISTS position_logs (
            pos_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            ply           INTEGER,
            feature_name  TEXT,
            feature_value REAL
        );
        CREATE TABLE IF NOT EXISTS baseline_evals (
            eval_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            checkpoint    INTEGER,
            wins          INTEGER,
            losses        INTEGER,
            draws         INTEGER
        );
    """)
    conn.commit()
    return conn


## Logging Helpers


In [17]:
def log_run(conn, run_id, depth, lr, move_limit, notes):
    conn.execute(
        "INSERT OR IGNORE INTO runs (run_id, started_at, depth, learning_rate, move_limit, notes) VALUES (?,?,?,?,?,?)",
        (run_id, datetime.now().isoformat(), depth, lr, move_limit, notes),
    )
    conn.commit()

def log_game(conn, run_id, game_num, outcome, n_halfmoves, duration_s, terminated_by):
    conn.execute(
        "INSERT INTO games (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by) VALUES (?,?,?,?,?,?)",
        (run_id, game_num, outcome, n_halfmoves, duration_s, terminated_by),
    )
    conn.commit()

def log_weights(conn, run_id, game_num, weights_dict):
    """weights_dict: {feature_name: value} — attribution weights for NN."""
    conn.executemany(
        "INSERT INTO weights (run_id, game_number, feature_name, weight_value) VALUES (?,?,?,?)",
        [(run_id, game_num, name, val) for name, val in weights_dict.items()],
    )
    conn.commit()

def log_weight_deltas(conn, run_id, game_num, deltas_dict):
    conn.executemany(
        "INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) VALUES (?,?,?,?)",
        [(run_id, game_num, name, val) for name, val in deltas_dict.items()],
    )
    conn.commit()

def log_game_record(conn, run_id, game_num, pgn, td_error):
    conn.execute(
        "INSERT INTO game_records (run_id, game_number, pgn, td_error) VALUES (?,?,?,?)",
        (run_id, game_num, pgn, td_error),
    )
    conn.commit()

def log_feature_stats(conn, run_id, game_num, positions, outcome):
    """Log per-game mean feature values from white's perspective."""
    white_feats = defaultdict(list)
    for feat_dict, color, *_ in positions:   # *_ handles both 2- and 3-tuple formats
        sign = 1.0 if color == chess.WHITE else -1.0
        for name, val in feat_dict.items():
            white_feats[name].append(sign * val)
    rows = [
        (run_id, game_num, outcome, name, sum(vals) / len(vals))
        for name, vals in white_feats.items() if vals
    ]
    conn.executemany(
        "INSERT INTO feature_stats (run_id, game_number, outcome, feature_name, mean_value) VALUES (?,?,?,?,?)",
        rows,
    )
    conn.commit()

def log_position_logs(conn, run_id, game_num, positions):
    rows = []
    for ply, (feat_dict, color, *_) in enumerate(positions):   # *_ handles both formats
        for name, val in feat_dict.items():
            rows.append((run_id, game_num, ply, name, val))
    conn.executemany(
        "INSERT INTO position_logs (run_id, game_number, ply, feature_name, feature_value) VALUES (?,?,?,?,?)",
        rows,
    )
    conn.commit()

def get_last_game_number(conn, run_id):
    row = conn.execute(
        "SELECT MAX(game_number) FROM games WHERE run_id = ?", (run_id,)
    ).fetchone()
    return row[0] if row[0] is not None else 0

def load_last_model(run_id, model_dir):
    """Load the most recent checkpoint .pt file for this run_id."""
    import glob
    pattern = os.path.join(model_dir, f"nn_run_{run_id}_game_*.pt")
    files = sorted(
        glob.glob(pattern),
        key=lambda p: int(p.split("_game_")[1].replace(".pt", ""))
    )
    if not files:
        raise FileNotFoundError(f"No checkpoint found for run_id={run_id} in {model_dir}")
    path = files[-1]
    print(f"Resuming from {path}")
    return load_model(path)


## Self-Play Game Function

Positions stored as `(feature_dict, color)` — same structure as linear conditions,
enabling identical `feature_stats` logging. Feature tensors are built in `td_update()`.


In [18]:
def play_game(model, depth, move_limit=200, epsilon=0.1, opponent_model=None):
    """
    Play one self-play game.

    If opponent_model is None (default), the current model plays both sides
    (original behaviour).  If opponent_model is provided, the learner is
    randomly assigned WHITE or BLACK; the opponent plays the other color with
    pure greedy search (no epsilon-greedy exploration).

    Positions are returned as (feature_dict, color, is_learner) 3-tuples so
    that td_update() can skip gradient updates on opponent half-moves while
    still using them as bootstrap targets.
    """
    board = chess.Board()
    game  = chess.pgn.Game()
    node  = game
    positions     = []
    terminated_by = "natural"

    # Randomly assign learner color when playing against a past checkpoint
    learner_color = None
    if opponent_model is not None:
        learner_color = chess.WHITE if random.random() < 0.5 else chess.BLACK

    while True:
        if board.is_game_over(claim_draw=True):
            terminated_by = "natural"
            break
        if len(positions) >= move_limit:
            terminated_by = "move_limit"
            break

        features   = extract_features(board)
        color      = board.turn
        is_learner = (opponent_model is None) or (color == learner_color)

        if is_learner:
            if random.random() < epsilon:
                move = random.choice(list(board.legal_moves))
            else:
                move, _ = get_best_move(board, model, depth=depth, randomize=True)
        else:
            with torch.no_grad():
                move, _ = get_best_move(board, opponent_model, depth=depth, randomize=True)

        if move is None:
            break

        node = node.add_variation(move)
        board.push(move)
        positions.append((features, color, is_learner))

    if terminated_by == "move_limit":
        outcome = "1/2-1/2"
    else:
        outcome = board.result(claim_draw=True)

    game.headers["Result"] = outcome
    return positions, outcome, terminated_by, str(game)


## TD(0) Learning with Adam

For each position t in the game:

- Non-terminal: target = −V(s_{t+1}) with stop-grad (bootstrap target as fixed constant)
- Terminal: target = actual outcome ∈ {+1, 0, −1}
- loss = `draw_lr_scale` × (target − V(s_t))²  during draws; (target − V(s_t))² otherwise

Each position gets one independent Adam step (online TD(0)).


In [19]:
def outcome_for_color(outcome, color):
    if outcome == "1-0":
        return  1.0 if color == chess.WHITE else -1.0
    elif outcome == "0-1":
        return -1.0 if color == chess.WHITE else  1.0
    return 0.0  # draw


def td_update(model, optimizer, positions, outcome, draw_lr_scale=0.1):
    """
    Online TD(0) update for the neural net evaluator.

    positions : list of (feature_dict, color, is_learner) 3-tuples from play_game().
    Only positions where is_learner=True receive gradient updates.  All
    positions are used to build bootstrap targets so the TD chain is intact
    even when opponent half-moves are interleaved.

    Returns: mean absolute TD error over learner positions.
    """
    is_draw = (outcome == "1/2-1/2")
    scale   = draw_lr_scale if is_draw else 1.0

    # Build feature tensors for ALL positions (needed for bootstrap targets)
    tensors = [
        torch.tensor([feat[name] for name in FEATURE_NAMES], dtype=torch.float32)
        for feat, _, _ in positions
    ]

    # Bootstrap targets: V(s_{t+1}) with no grad (treated as fixed constants)
    with torch.no_grad():
        next_values = [model(x) for x in tensors[1:]]

    total_error = 0.0
    n_learner   = 0

    for t, (feat, color, is_learner) in enumerate(positions):
        if not is_learner:
            continue   # opponent position — no gradient update

        if t < len(positions) - 1:
            target = -next_values[t]                          # perspective flip; detached
        else:
            target = torch.tensor(outcome_for_color(outcome, color))

        optimizer.zero_grad()
        v    = model(tensors[t])
        loss = scale * (target - v) ** 2
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            total_error += abs((target - v).item())
        n_learner += 1

    return total_error / n_learner if n_learner > 0 else 0.0


## Training Loop

In [20]:
def train(
    run_id,
    n_games,
    depth                     = 3,
    learning_rate             = 0.001,
    move_limit                = 200,
    checkpoint_every          = 10,
    db_path                   = "../../data/chess_learning.db",
    model_dir                 = "../../models",
    notes                     = "",
    resume                    = False,
    log_positions             = True,
    epsilon                   = 0.1,
    draw_lr_scale             = 0.1,
    hidden_size               = 32,
    verbose                   = True,
    past_opponent_freq        = 0.0,
    past_opponent_checkpoints = None,
):
    """
    Main training loop for the neural net condition.

    past_opponent_freq (float): probability [0, 1] that any given game uses a
        fixed past checkpoint as the opponent rather than the current model.
        0.0 = pure self-play (original behaviour).
    past_opponent_checkpoints (list[int] | None): game numbers whose saved .pt
        files to load as fixed opponents.  E.g. [100, 200].  Missing files are
        silently skipped.
    """
    conn = setup_database(db_path)

    if resume:
        model      = load_last_model(run_id, model_dir)
        start_game = get_last_game_number(conn, run_id) + 1
        print(f"Resuming run_id={run_id} from game {start_game}")
    else:
        model      = new_model(hidden_size=hidden_size)
        start_game = 1
        log_run(conn, run_id, depth, learning_rate, move_limit, notes)
        log_weights(conn, run_id, 0, attribution_weights(model))  # game-0 checkpoint

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # ── Load past checkpoint models (once, before the training loop) ──
    past_models = {}
    if past_opponent_freq > 0 and past_opponent_checkpoints:
        for ckpt in past_opponent_checkpoints:
            pt_path = os.path.join(model_dir, f"nn_run_{run_id}_game_{ckpt}.pt")
            if os.path.exists(pt_path):
                m = load_model(pt_path)
                m.eval()
                past_models[ckpt] = m
                print(f"Loaded past checkpoint: game {ckpt}")
            else:
                print(f"Past checkpoint not found, skipping: {pt_path}")
        if not past_models:
            print("Warning: no past checkpoint files found; running pure self-play.")

    past_model_list = list(past_models.values())  # for fast random.choice

    outcomes  = {"1-0": 0, "0-1": 0, "1/2-1/2": 0}
    n_past    = 0   # count of games played against a past checkpoint
    t_start   = time.time()
    attr_prev = attribution_weights(model)

    for i in range(n_games):
        game_num = start_game + i
        t0       = time.time()

        # Decide whether to use a past checkpoint as the opponent this game
        opponent_model = None
        if past_model_list and random.random() < past_opponent_freq:
            opponent_model = random.choice(past_model_list)
            n_past += 1

        positions, outcome, terminated_by, pgn = play_game(
            model, depth, move_limit, epsilon=epsilon,
            opponent_model=opponent_model,
        )
        td_error = td_update(
            model, optimizer, positions, outcome, draw_lr_scale=draw_lr_scale
        )

        attr_now  = attribution_weights(model)
        deltas    = {name: attr_now[name] - attr_prev[name] for name in FEATURE_NAMES}
        attr_prev = attr_now

        elapsed = time.time() - t0
        log_game(conn, run_id, game_num, outcome, len(positions), elapsed, terminated_by)
        log_game_record(conn, run_id, game_num, pgn, td_error)
        log_weight_deltas(conn, run_id, game_num, deltas)
        log_feature_stats(conn, run_id, game_num, positions, outcome)
        if log_positions:
            log_position_logs(conn, run_id, game_num, positions)

        outcomes[outcome] = outcomes.get(outcome, 0) + 1

        if game_num % checkpoint_every == 0:
            log_weights(conn, run_id, game_num, attr_now)
            model_path = os.path.join(model_dir, f"nn_run_{run_id}_game_{game_num}.pt")
            save_model(model, model_path)

        if verbose and game_num % 10 == 0:
            elapsed_total = time.time() - t_start
            rate = 60 * i / elapsed_total if elapsed_total > 0 else 0
            d, b, dr = outcomes["1-0"], outcomes["0-1"], outcomes["1/2-1/2"]
            total = d + b + dr
            dec_pct  = 100 * (d + b) / total if total > 0 else 0
            past_pct = 100 * n_past / (i + 1) if i >= 0 else 0
            print(
                f"Game {game_num:>5} | 1-0: {d:>4}  0-1: {b:>4}  Draw: {dr:>4} "
                f"({dec_pct:.1f}% decisive) | TD err: {td_error:.4f} "
                f"| Past opp: {past_pct:.0f}% | {rate:.1f} games/min"
            )

    # Final checkpoint if not already on a boundary
    final_game = start_game + n_games - 1
    if final_game % checkpoint_every != 0:
        final_attr = attribution_weights(model)
        log_weights(conn, run_id, final_game, final_attr)
        model_path = os.path.join(model_dir, f"nn_run_{run_id}_game_{final_game}.pt")
        save_model(model, model_path)

    conn.close()
    d, b, dr = outcomes["1-0"], outcomes["0-1"], outcomes["1/2-1/2"]
    print(f"\nDone. {n_games} games in {(time.time()-t_start)/60:.1f} min.")
    print(f"1-0: {d}  0-1: {b}  Draw: {dr}  ({100*(d+b)/(d+b+dr):.1f}% decisive)")
    if past_models:
        print(f"Past-opponent games: {n_past}/{n_games} ({100*n_past/n_games:.0f}%)")
    return model


## Configuration & Run

Set `RESUME = False` to start a new run; `True` to continue an existing run.
Model checkpoints: `models/nn_run_{RUN_ID}_game_{N}.pt`

**Past checkpoint opponent** (`PAST_OPPONENT_FREQ`, `PAST_OPPONENT_CHECKPOINTS`):
Set `PAST_OPPONENT_FREQ > 0` to play occasionally against a frozen earlier version
of the network. For example, `PAST_OPPONENT_FREQ = 0.2` with
`PAST_OPPONENT_CHECKPOINTS = [100, 200]` means ~20% of games pit the current
model against a randomly selected frozen snapshot (game 100 or game 200).
The frozen model does not receive any gradient updates. This diversifies the
training distribution and reduces self-play collapse.


In [21]:
RUN_ID           = 24
N_GAMES          = 300           # games this session
DEPTH            = 3
LEARNING_RATE    = 0.001        # Adam LR
MOVE_LIMIT       = 200
CHECKPOINT_EVERY = 10
HIDDEN_SIZE      = 32           # 19 -> 32 -> 1
EPSILON          = 0.1          # epsilon-greedy exploration (learner only)
DRAW_LR_SCALE    = 0.1          # global loss scale during draw games
DB_PATH          = "../../data/chess_learning.db"
MODEL_DIR        = "../../models"
ENGINE_VERSION   = "v2_neural_net_h32"
RESUME           = True

# Past-checkpoint opponent — set PAST_OPPONENT_FREQ > 0 to activate
PAST_OPPONENT_FREQ        = 0.2   # fraction of games using a past checkpoint as opponent
PAST_OPPONENT_CHECKPOINTS = [100, 200]   # game numbers to use as fixed opponents

model = train(
    run_id                    = RUN_ID,
    n_games                   = N_GAMES,
    depth                     = DEPTH,
    learning_rate             = LEARNING_RATE,
    move_limit                = MOVE_LIMIT,
    checkpoint_every          = CHECKPOINT_EVERY,
    db_path                   = DB_PATH,
    model_dir                 = MODEL_DIR,
    notes                     = ENGINE_VERSION,
    resume                    = RESUME,
    epsilon                   = EPSILON,
    draw_lr_scale             = DRAW_LR_SCALE,
    hidden_size               = HIDDEN_SIZE,
    past_opponent_freq        = PAST_OPPONENT_FREQ,
    past_opponent_checkpoints = PAST_OPPONENT_CHECKPOINTS,
)


Resuming from ../../models\nn_run_24_game_1005.pt
Resuming run_id=24 from game 1006
Loaded past checkpoint: game 100
Loaded past checkpoint: game 200
Game  1010 | 1-0:    1  0-1:    2  Draw:    2 (60.0% decisive) | TD err: 0.0695 | Past opp: 0% | 0.4 games/min
Game  1020 | 1-0:    5  0-1:    6  Draw:    4 (73.3% decisive) | TD err: 0.0867 | Past opp: 7% | 0.5 games/min
Game  1030 | 1-0:    9  0-1:   11  Draw:    5 (80.0% decisive) | TD err: 0.0860 | Past opp: 16% | 0.5 games/min
Game  1040 | 1-0:   10  0-1:   16  Draw:    9 (74.3% decisive) | TD err: 0.0673 | Past opp: 17% | 0.5 games/min
Game  1050 | 1-0:   13  0-1:   19  Draw:   13 (71.1% decisive) | TD err: 0.0611 | Past opp: 22% | 0.5 games/min
Game  1060 | 1-0:   18  0-1:   21  Draw:   16 (70.9% decisive) | TD err: 0.0846 | Past opp: 18% | 0.5 games/min
Game  1070 | 1-0:   22  0-1:   25  Draw:   18 (72.3% decisive) | TD err: 0.0710 | Past opp: 17% | 0.6 games/min
Game  1080 | 1-0:   24  0-1:   29  Draw:   22 (70.7% decisive) | TD 

## Attribution Weight Trajectory

Quick check: read attribution snapshots from the DB and display how each feature's
local sensitivity (dV/dx_i at starting position) evolved over training.
Full visualization in `03_visualize.qmd`.


In [22]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    "SELECT game_number, feature_name, weight_value FROM weights "
    "WHERE run_id = ? ORDER BY game_number, feature_name",
    (RUN_ID,)
).fetchall()
conn.close()

by_game = defaultdict(dict)
for game_num, feature, val in rows:
    by_game[game_num][feature] = val

game_nums = sorted(by_game.keys())
header = f"{'Attribution (dV/dx_i)':<24}" + "".join(f"{g:>10}" for g in game_nums)
print(header)
print("-" * len(header))
for feature in FEATURE_NAMES:
    row = f"{feature:<24}" + "".join(f"{by_game[g].get(feature, 0):>10.4f}" for g in game_nums)
    print(row)


Attribution (dV/dx_i)            0        10        20        30        40        50        60        70        80        90       100       110       120       130       140       150       160       170       180       190       200       210       220       230       240       250       260       270       280       290       300       310       320       330       340       344       350       360       370       380       390       400       410       420       430       440       444       450       460       470       480       490       500       510       520       530       540       544       550       560       570       580       590       600       610       620       630       640       644       650       660       670       680       690       700       710       720       730       740       750       760       770       780       790       800       810       820       830       840       850       860       870       880       890       894       900       910      